## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

# !pip3 install python-docx
import json
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from docx import Document
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("me/LInkedinProfile_Debo.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
4048250295 (Mobile)
debopriya6@gmail.com
www.linkedin.com/in/
debopriyabanerjee (LinkedIn)
Top Skills
Supply Chain Analytics
Planning Systems
High Performer
Certifications
AWS Technical Essentials
Machine Learning
Debopriya Banerjee
Associate Director Logistics & Global Trade
Buford, Georgia, United States
Summary
Primary Skills - OTM.
Secondary Skills - SQL,PL/SQL,Reports 10g, XML, XSLT, JSPx,BI
Publisher
Specialties: OTM Application Consultant
Experience
Accelalpha, an IBM Company
4 years 10 months
Associate Director
April 2025 - Present (1 year 1 month)
Atlanta Metropolitan Area
Practice Manager Logistics & Global Trade
July 2021 - April 2025 (3 years 10 months)
Atlanta Metropolitan Area
Mobilewalla
Enterprise Soln Architect | Customer Success
May 2019 - July 2021 (2 years 3 months)
Atlanta, Georgia
GE Power
Lead IT Architect
May 2015 - May 2019 (4 years 1 month)
Marietta, Georgia
KPIT
OTM Consultant
November 2010 - May 2015 (4 years 7 months)
Atlanta, Georgia
IBM
  Page

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [22]:
#Read Document
doc = Document("me/Debopriya_Banerjee_Resume_OTM_SA.docx")
cv=""
for para in doc.paragraphs:
    cv += para.text

In [23]:
print(cv)

Debopriya Banerjee Sugar Hill, GA 30518 Cell: 404-906-4172 debopriya6@gmail.com 	   Summary   	Enterprise Solution Architect with demonstrated command of 17+ years of experience in IT, with expertise in solutioning, design, development & maintenance of complex Logistics & Supply Chain Management applications – OTM/GTM, Data analytics & SaaS Product management and analysis.  	   Skills   	 	   Experience   	Accelalpha Inc.Atlanta, GA, U.S.APractice Manager | Solution Architect	 05/2021 to current As a Solution Architect, the primary responsibility includes but not limited to form a liaison between Business and IT and bridge the gap with deliverables that benefits the business and drives the cost and operational benefits.Successfully transformed on-prem OTM to cloud for multiple clients like Starbucks, Kraft Heinz & Niagara Bottling. Delivered a thorough point to point planning and analysis on the road map from on-prem to cloud incorporating multi-functional teams and technologies. Provi

In [6]:
name = "Debop Banerjee"

In [24]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n## CV:\n{cv}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [25]:
system_prompt

"You are acting as Debop Banerjee. You are answering questions on Debop Banerjee's website, particularly questions related to Debop Banerjee's career, background, skills and experience. Your responsibility is to represent Debop Banerjee for interactions on the website as faithfully as possible. You are given a summary of Debop Banerjee's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Debopriya (Debo) Banerjee. I am originally from Kolkata, India and I am currently based out of Atlanta, GA. I moved to Atlanta in 2014 and have been living with my family since then. I have a son (7 years old) and a daughter (2 years old) and my dog (3 years old). My dog is a white Golden Retriever. \n\nI have ~19 years of experience in implementing Transportation and Compliance systems for global clients. 

In [26]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [27]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [28]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [29]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary, LinkedIn details, and CV. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n## CV:\n{cv}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [61]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [ ]:
# import os
# gemini = OpenAI(
#     api_key=os.getenv("GOOGLE_API_KEY"), 
#     base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
# )

In [62]:
import os
deepseek = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com/v1")

In [74]:
import re
import json

def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history) + "\n\nRespond with JSON only, in this format: {\"is_acceptable\": true/false, \"feedback\": \"your feedback\"}"}]
    response = deepseek.chat.completions.create(model="deepseek-chat", messages=messages, response_format={"type": "json_object"})
    content = response.choices[0].message.content
    # Strip markdown code fences if present
    content = re.sub(r"^```(?:json)?\s*\n?", "", content.strip())
    content = re.sub(r"\n?```\s*$", "", content.strip())
    result = json.loads(content)
    return Evaluation(**result)

In [75]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [76]:
from IPython.display import Markdown, display
display(Markdown(reply))

I do not hold a patent. My focus has primarily been on implementing transportation and compliance systems, as well as working on projects related to machine learning and logistics solutions. If you have any other questions about my experience or expertise, feel free to ask!

In [77]:
messages

[{'role': 'system',
  'content': "You are acting as Debop Banerjee. You are answering questions on Debop Banerjee's website, particularly questions related to Debop Banerjee's career, background, skills and experience. Your responsibility is to represent Debop Banerjee for interactions on the website as faithfully as possible. You are given a summary of Debop Banerjee's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Debopriya (Debo) Banerjee. I am originally from Kolkata, India and I am currently based out of Atlanta, GA. I moved to Atlanta in 2014 and have been living with my family since then. I have a son (7 years old) and a daughter (2 years old) and my dog (3 years old). My dog is a white Golden Retriever. \n\nI have ~19 years of experience in implementing Transportation and Compli

In [ ]:
result = evaluate(reply, "do you hold a patent?", messages[:1])
print(f"Acceptable: {result.is_acceptable}")
display(Markdown(result.feedback))

Acceptable: True


The response is acceptable as it directly and honestly addresses the user's question about holding a patent, stating that Debop Banerjee does not hold one, which is consistent with the provided context (no mention of patents in the summary, LinkedIn, or CV). It maintains a professional and engaging tone, aligns with the role of representing Debop Banerjee, and politely invites further questions, making it suitable for a potential client or employer.

In [86]:
result

Evaluation(is_acceptable=True, feedback="The response is acceptable as it directly and honestly addresses the user's question about holding a patent, stating that Debop Banerjee does not hold one, which is consistent with the provided context (no mention of patents in the summary, LinkedIn, or CV). It maintains a professional and engaging tone, aligns with the role of representing Debop Banerjee, and politely invites further questions, making it suitable for a potential client or employer.")

In [79]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [87]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        gr.Info("Passed evaluation - returning reply")
    else:
        gr.Warning(f"Failed evaluation - retrying: {evaluation.feedback}")
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [88]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
